In [0]:
import requests, json
from datetime import datetime, timezone

LAT, LON = 28.6139, 77.2090
WEATHER_VARS = "temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,precipitation,boundary_layer_height"
now_utc = datetime.now(timezone.utc)

def fetch(url, hourly):
    r = requests.get(
        url,
        params={"latitude": LAT, "longitude": LON, "hourly": hourly,
                "past_days": 3, "forecast_days": 1, "timezone": "Asia/Kolkata"},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

weather = fetch("https://api.open-meteo.com/v1/forecast", WEATHER_VARS)
air_quality = fetch("https://air-quality-api.open-meteo.com/v1/air-quality", "pm2_5")

rows = [
    ("weather", now_utc, json.dumps(weather)),
    ("air_quality", now_utc, json.dumps(air_quality)),
]
df = spark.createDataFrame(rows, schema="source string, ingested_at timestamp, raw_json string")
df.write.mode("append").saveAsTable("workspace.delhi_air.bronze_openmeteo")
print("saved", len(rows), "rows")